# LeetCode 547 — Number of Provinces bằng Python

Link đề: https://leetcode.com/problems/number-of-provinces/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Union Find, Graph

Skeleton LeetCode:

```python
class Solution:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        ...
```

Ghi chú: đây là bài "đếm **thành phần liên thông**" (connected components) kinh điển của đồ thị. Notebook đi từ cách gộp nhóm thủ công, qua DFS, BFS, Union-Find, đến bao đóng bắc cầu (Warshall), rồi so sánh các cách với nhau.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho ma trận `isConnected` cỡ `n x n`, đếm có bao nhiêu **tỉnh** (nhóm thành phố nối với nhau trực tiếp hoặc gián tiếp).
2. Nắm các công cụ Python cần thiết: ma trận là list lồng list, `set` (hợp, giao), `collections.deque` (hàng đợi), đệ quy, và cấu trúc **Union-Find** (còn gọi DSU, Disjoint Set Union).
3. Đi từ **gộp nhóm bằng set** → **DFS** → **BFS** → **Union-Find** → **bao đóng bắc cầu Warshall** và biết mỗi cách khác nhau ở đâu.
4. Hiểu mấu chốt: "quan hệ bắc cầu" (a nối b, b nối c thì a nối c gián tiếp) biến mỗi tỉnh thành một **cụm**; mỗi lần gặp một thành phố **chưa thăm**, ta gặp đúng **một tỉnh mới**.
5. Tránh các lỗi kinh điển: đếm số dòng khác nhau của ma trận (không xét bắc cầu), quên mảng `visited` gây đệ quy vô hạn, giảm bộ đếm khi hai thành phố đã cùng gốc, và nhầm `isConnected` với danh sách cạnh.

## 2. Đề bài & ví dụ

Có `n` thành phố. Một số cặp thành phố nối trực tiếp với nhau, số còn lại thì không. Nếu thành phố `a` nối trực tiếp với `b`, và `b` nối trực tiếp với `c`, thì `a` nối **gián tiếp** với `c`.

Một **tỉnh** (province) là một nhóm thành phố nối với nhau trực tiếp hoặc gián tiếp, và **không có thành phố nào ngoài nhóm** cũng nối với nhóm đó.

Cho ma trận `isConnected` cỡ `n x n`, trong đó `isConnected[i][j] = 1` nếu thành phố `i` và `j` nối trực tiếp, `0` nếu không. Trả về **tổng số tỉnh**.

**Ví dụ 1**

- Input: `isConnected = [[1,1,0],[1,1,0],[0,0,1]]`
- Output: `2` (thành phố 0 và 1 chung một tỉnh; thành phố 2 một mình một tỉnh)

**Ví dụ 2**

- Input: `isConnected = [[1,0,0],[0,1,0],[0,0,1]]`
- Output: `3` (không cặp nào nối nhau, mỗi thành phố là một tỉnh)

**Ràng buộc:**

- `1 <= n <= 200`
- `n == isConnected.length == isConnected[i].length`
- `isConnected[i][j]` là `1` hoặc `0`
- `isConnected[i][i] == 1` (thành phố luôn nối với chính nó)
- `isConnected[i][j] == isConnected[j][i]` (ma trận đối xứng)

Hệ quả: `n` nhỏ (tối đa 200) nên một lời giải `O(n^2)` đọc hết ma trận là hoàn toàn đủ nhanh; thậm chí `O(n^3)` cũng chỉ ~8 triệu bước.

In [1]:
from typing import List
from collections import deque
import random
import time

ex1 = [[1, 1, 0], [1, 1, 0], [0, 0, 1]]
ex2 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]


def print_matrix(M):
    for row in M:
        print(" ".join(map(str, row)))


for name, M, expected in [("Ví dụ 1", ex1, 2), ("Ví dụ 2", ex2, 3)]:
    print(name, "- n =", len(M), "- kỳ vọng:", expected)
    print_matrix(M)
    print()

Ví dụ 1 - n = 3 - kỳ vọng: 2
1 1 0
1 1 0
0 0 1

Ví dụ 2 - n = 3 - kỳ vọng: 3
1 0 0
0 1 0
0 0 1



## 3. Kiến thức nền

### 3.1. Ma trận kề (adjacency matrix): đọc `isConnected[i][j]`

Trong Python, ma trận là **list chứa các list**. `M[i]` là dòng `i`, `M[i][j]` là ô ở dòng `i`, cột `j`. Ở bài này, ô `M[i][j] == 1` nghĩa là "thành phố `i` và `j` nối trực tiếp". Nên **dòng `i` chính là danh sách "bạn trực tiếp" của thành phố `i`** (dưới dạng 0/1).

`enumerate(M[i])` cho ta cặp `(j, giá trị)` để lấy ra các `j` thỏa `M[i][j] == 1`.

In [2]:
M = [[1, 1, 0, 0],
     [1, 1, 1, 0],
     [0, 1, 1, 0],
     [0, 0, 0, 1]]
print("M[1] =", M[1])
print("M[1][2] =", M[1][2], "-> thành phố 1 và 2 nối trực tiếp")
print("M[0][2] =", M[0][2], "-> thành phố 0 và 2 KHÔNG nối trực tiếp")
for i in range(len(M)):
    friends = [j for j, x in enumerate(M[i]) if x == 1 and j != i]
    print(f"bạn trực tiếp của {i}:", friends)

M[1] = [1, 1, 1, 0]
M[1][2] = 1 -> thành phố 1 và 2 nối trực tiếp
M[0][2] = 0 -> thành phố 0 và 2 KHÔNG nối trực tiếp
bạn trực tiếp của 0: [1]
bạn trực tiếp của 1: [0, 2]
bạn trực tiếp của 2: [1]
bạn trực tiếp của 3: []


### 3.2. Đồ thị và thành phần liên thông

Ta có thể vẽ mỗi thành phố là một **đỉnh** (node), mỗi cặp nối trực tiếp là một **cạnh** (edge). Khi đó ma trận `isConnected` chính là một **đồ thị vô hướng** (cạnh không có chiều).

Trong ví dụ ở 3.1: cạnh `0-1`, `1-2`. Thành phố `0` và `2` không nối trực tiếp, nhưng nối **gián tiếp** qua `1`. Vậy `{0, 1, 2}` là một tỉnh, còn `{3}` là một tỉnh khác. Nhóm đỉnh như vậy gọi là **thành phần liên thông** (connected component): đi từ đỉnh này sang đỉnh kia trong nhóm được bằng các cạnh, và không có cạnh nào đi ra ngoài nhóm.

Vậy bài toán = **đếm số thành phần liên thông của đồ thị**.

### 3.3. `set`: hợp, giao, kiểm tra rời nhau

`set` là tập hợp không trùng, tra cứu nhanh. Ba phép ta sẽ dùng cho cách gộp nhóm:

- `a | b` (hợp): mọi phần tử của `a` hoặc `b`.
- `a & b` (giao): phần tử có ở cả hai. Nếu giao **rỗng** thì hai tập **rời nhau**; `bool(a & b)` là `True` khi có phần tử chung.
- `a |= b`: gộp `b` vào `a`.

In [3]:
a = {0, 1}
b = {1, 2}
c = {5}
print("a | b =", a | b)
print("a & b =", a & b, "-> có chung?", bool(a & b))
print("a & c =", a & c, "-> có chung?", bool(a & c))
a |= b
print("sau a |= b:", a)

a | b = {0, 1, 2}
a & b = {1} -> có chung? True
a & c = set() -> có chung? False
sau a |= b: {0, 1, 2}


### 3.4. `deque` và vòng lặp `while` cho BFS

`collections.deque` là hàng đợi hai đầu: `append(x)` thêm vào cuối, `popleft()` lấy ra từ đầu, cả hai đều `O(1)`. (`list.pop(0)` thì `O(n)` vì phải dịch cả list.) BFS dùng deque để xử lý "ai vào trước thì ra trước" (FIFO). Còn nếu dùng `list` với `append`/`pop()` (lấy từ cuối) thì ta có **ngăn xếp** (stack, LIFO) — nền tảng của DFS không đệ quy.

In [4]:
q = deque([0])
q.append(1)
q.append(2)
print(q)
print("popleft ->", q.popleft(), "| còn", list(q))

stack = [0, 1, 2]
print("pop ->", stack.pop(), "| còn", stack)

deque([0, 1, 2])
popleft -> 0 | còn [1, 2]
pop -> 2 | còn [0, 1]


### 3.5. Đệ quy và mảng `visited`

Một hàm **đệ quy** là hàm tự gọi lại chính nó. Khi duyệt đồ thị mà không "ghi nhớ đã đến chỗ nào", ta sẽ đi vòng quanh mãi (`0 -> 1 -> 0 -> 1 ...`). Giải pháp: dùng mảng `visited` (hay `vis`), đánh dấu `vis[i] = True` ngay khi đến thành phố `i`, và **không đi vào đỉnh đã đánh dấu**.

Ví dụ ngược dòng: nếu bỏ `vis`, DFS trên hai thành phố nối nhau sẽ gọi qua lại vô hạn và Python báo `RecursionError`.

In [5]:
def broken_dfs_demo(M):
    """KHÔNG có vis nên đệ quy mãi (chỉ để minh họa; có chặn độ sâu cho an toàn)."""
    depth_log = []

    def dfs(i, depth):
        if depth > 5:                   # chặn tay để notebook không chạy vô hạn
            return
        depth_log.append(i)
        for j in range(len(M)):
            if M[i][j] and j != i:      # bỏ qua i == j nhưng vẫn quay đầu 0 <-> 1 mãi
                dfs(j, depth + 1)

    dfs(0, 0)
    return depth_log


print(broken_dfs_demo([[1, 1], [1, 1]]))   # 0,1,0,1,0,1 ... lặp mãi

[0, 1, 0, 1, 0, 1]


### 3.6. Union-Find (DSU) trong 6 dòng

**Union-Find** quản lý các nhóm rời nhau bằng một mảng `parent`. Mỗi nhóm có một **gốc** (root) đại diện, `parent[x]` trỏ lên cha của `x`; gốc thì `parent[root] == root`.

- `find(x)`: đi lên từ `x` đến gốc. Ta "nén đường đi" (path compression) để lần sau nhanh hơn.
- `union(a, b)`: nối gốc của `a` vào gốc của `b` để hai nhóm thành một.

Lúc đầu mỗi người là một nhóm: `parent = [0, 1, 2, 3]`.

In [6]:
parent = [0, 1, 2, 3]


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # trỏ thẳng lên ông (nén đường đi một nửa)
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        return True      # có gộp thật
    return False         # đã cùng nhóm


print(union(0, 1), parent)   # True  [1, 1, 2, 3]
print(union(1, 2), parent)   # True  [1, 2, 2, 3]
print(union(0, 2), parent)   # False: 0 và 2 đã cùng gốc 2
print("gốc của 0:", find(0), "| gốc của 3:", find(3))

True [1, 1, 2, 3]
True [1, 2, 2, 3]
False [2, 2, 2, 3]
gốc của 0: 2 | gốc của 3: 3


## 4. Ý tưởng nền tảng / trực giác

Hãy làm tay với ví dụ: 6 thành phố, nối `0-2`, `2-4`, `1-3`, còn `5` đứng một mình.

```
0 -- 2 -- 4        1 -- 3        5
```

Đọc bằng mắt: tỉnh thứ nhất `{0, 2, 4}`, tỉnh thứ hai `{1, 3}`, tỉnh thứ ba `{5}` → **3 tỉnh**.

Nhưng chú ý: **`0` và `4` không nối trực tiếp** (`M[0][4] = 0`), vẫn cùng một tỉnh nhờ `2`. Nếu chỉ nhìn "dòng nào giống dòng nào" ta sẽ sai (xem lỗi 1 ở mục 13).

Cách người ta làm tự nhiên: đứng ở thành phố `0` (chưa ai thăm), đi sang tất cả bạn của `0`, rồi bạn của bạn... "loang" đến khi hết đường; nhóm vừa loang được là **một tỉnh**. Sau đó tìm một thành phố chưa được loang tới, lặp lại. Số lần ta phải **bắt đầu loang mới** chính là số tỉnh.

In [7]:
n = 6
edges = [(0, 2), (2, 4), (1, 3)]
M6 = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
for a, b in edges:
    M6[a][b] = M6[b][a] = 1

print_matrix(M6)
print("M6[0][4] =", M6[0][4], "(không nối trực tiếp)")
print("bạn của 0:", [j for j in range(n) if M6[0][j] and j != 0])
print("bạn của 2:", [j for j in range(n) if M6[2][j] and j != 2])
print("bạn của 4:", [j for j in range(n) if M6[4][j] and j != 4])
print("=> 0 -> 2 -> 4: 0 và 4 cùng tỉnh dù không nối trực tiếp")

1 0 1 0 0 0
0 1 0 1 0 0
1 0 1 0 1 0
0 1 0 1 0 0
0 0 1 0 1 0
0 0 0 0 0 1
M6[0][4] = 0 (không nối trực tiếp)
bạn của 0: [2]
bạn của 2: [0, 4]
bạn của 4: [2]
=> 0 -> 2 -> 4: 0 và 4 cùng tỉnh dù không nối trực tiếp


## 5. Cách 1 — Brute Force: gộp nhóm bằng `set`

### Ý tưởng

Giữ một danh sách `groups`, mỗi phần tử là một `set` các thành phố đang biết là cùng tỉnh. Duyệt mọi cặp `(i, j)` với `i <= j`; nếu `M[i][j] == 1`:

1. Tạo `merged = {i, j}`.
2. Duyệt các nhóm đang có: nhóm nào **giao** với `merged` (có chung phần tử) thì gộp vào `merged`; nhóm nào không thì giữ nguyên.
3. Đưa `merged` vào danh sách.

Lấy `j` từ `i` (kể cả `i == j`) để thành phố đứng một mình (chỉ có `M[i][i] = 1`) cũng tự thành một nhóm. Đáp án = `len(groups)`.

### Code

In [8]:
def count_groups_merge(M: List[List[int]]) -> int:
    n = len(M)
    groups = []                      # list các set
    for i in range(n):
        for j in range(i, n):
            if M[i][j]:
                merged = {i, j}
                rest = []
                for g in groups:
                    if g & merged:       # có chung thành phố -> cùng tỉnh
                        merged |= g
                    else:
                        rest.append(g)
                rest.append(merged)
                groups = rest
    return len(groups)


def groups_merge(M):
    """Bản trả về chính các nhóm để nhìn tận mắt."""
    n = len(M)
    groups = []
    for i in range(n):
        for j in range(i, n):
            if M[i][j]:
                merged = {i, j}
                rest = []
                for g in groups:
                    if g & merged:
                        merged |= g
                    else:
                        rest.append(g)
                rest.append(merged)
                groups = rest
    return groups


print(count_groups_merge(ex1), count_groups_merge(ex2), count_groups_merge(M6))
print(groups_merge(M6))

2 3 3
[{1, 3}, {0, 2, 4}, {5}]


### Độ phức tạp và nút thắt

- Có `O(n^2)` ô để xét. Mỗi ô có `M[i][j] = 1` có thể phải so với tối đa `O(n)` nhóm, mỗi lần giao/hợp set tốn tới `O(n)` → tổng tệ nhất khoảng `O(n^3)` thời gian, `O(n)` bộ nhớ cho các nhóm.
- Nút thắt: ta **tìm nhóm bằng cách quét mọi nhóm** mỗi lần. Đáng lẽ chỉ cần biết "thành phố này thuộc nhóm nào" trong `O(1)`.

Hai hướng cải tiến: (a) **loang** từ một thành phố để vét sạch cả tỉnh một lần (DFS/BFS), (b) **tra nhóm nhanh** bằng mảng cha (Union-Find).

## 6. Ý tưởng thuật toán / bước đột phá

Thay vì ghép nhóm từ các cạnh, ta **đi tìm nhóm**:

> Duyệt thành phố `0, 1, ..., n-1`. Nếu thành phố `i` **chưa thăm**, thì nó thuộc một tỉnh chưa từng được đếm: tăng đáp án thêm 1, rồi **đánh dấu thăm toàn bộ tỉnh của `i`** (bằng loang). Thành phố đã thăm thì bỏ qua vì tỉnh của nó đã được đếm.

Vì sao mỗi tỉnh chỉ đếm đúng một lần? Khi ta loang từ `i`, ta đánh dấu **mọi** thành phố đến được từ `i` — đó đúng là cả tỉnh của `i`. Về sau, những thành phố đó không còn "chưa thăm" nên không tạo ra lần đếm mới.

Điểm duy nhất cần làm là bước "loang". Có hai cách tự nhiên: **DFS** (đào sâu bằng đệ quy/ngăn xếp) và **BFS** (loang từng lớp bằng hàng đợi). Cả hai đều chạm mỗi thành phố đúng một lần và đọc một dòng ma trận cho mỗi thành phố → `O(n^2)`.

In [9]:
# Phiên bản "khung" để thấy rõ bước đột phá: chỉ khác nhau ở hàm spread(i)
def count_by_spreading(M, spread):
    n = len(M)
    vis = [False] * n
    ans = 0
    for i in range(n):
        if not vis[i]:
            ans += 1                  # gặp một tỉnh mới
            spread(i, vis)            # đánh dấu cả tỉnh của i
    return ans


def spread_dfs_for_M6(i, vis):
    vis[i] = True
    for j in range(len(M6)):
        if M6[i][j] and not vis[j]:
            spread_dfs_for_M6(j, vis)


print(count_by_spreading(M6, spread_dfs_for_M6))   # 3

3


## 7. Các cách tiếp theo

Tổng cộng ta có 5 cách: (1) gộp nhóm bằng set (mục 5), (2) DFS đệ quy, (3) BFS, (4) Union-Find, (5) bao đóng bắc cầu Warshall.

### 7.1. Cách 2 — DFS đệ quy

Giải thích từng đoạn:

1. `vis = [False] * n`: chưa thành phố nào được thăm.
2. Hàm con `dfs(i)`: đánh dấu `vis[i] = True`, rồi với mỗi `j` có `M[i][j] == 1` mà **chưa thăm**, gọi `dfs(j)`.
3. Vòng ngoài: gặp `i` chưa thăm thì `dfs(i)` và `ans += 1`.

Độ sâu đệ quy tối đa là `n = 200` (khi đồ thị là một đường thẳng), thấp hơn nhiều so với giới hạn mặc định ~1000 của Python nên an toàn.

In [10]:
def count_dfs(M: List[List[int]]) -> int:
    n = len(M)
    vis = [False] * n

    def dfs(i):
        vis[i] = True
        for j in range(n):
            if M[i][j] and not vis[j]:
                dfs(j)

    ans = 0
    for i in range(n):
        if not vis[i]:
            dfs(i)
            ans += 1
    return ans


print(count_dfs(ex1), count_dfs(ex2), count_dfs(M6))

2 3 3


In [11]:
# Code nộp LeetCode cho cách 2 (DFS)
class SolutionDFS:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        vis = [False] * n

        def dfs(i: int) -> None:
            vis[i] = True
            for j, x in enumerate(isConnected[i]):
                if x and not vis[j]:
                    dfs(j)

        ans = 0
        for i in range(n):
            if not vis[i]:
                dfs(i)
                ans += 1
        return ans


print(SolutionDFS().findCircleNum(ex1), SolutionDFS().findCircleNum(ex2))

2 3


**Độ phức tạp:** mỗi thành phố gọi `dfs` đúng một lần (nhờ `vis`) và mỗi lần đọc cả một dòng `n` ô → thời gian `O(n^2)`. Bộ nhớ `O(n)` cho `vis` và ngăn xếp đệ quy.

### 7.2. Cách 3 — BFS (loang từng lớp, không đệ quy)

Thay đệ quy bằng hàng đợi `deque`:

1. Gặp `i` chưa thăm: `ans += 1`, đánh dấu `vis[i] = True` rồi đưa `i` vào hàng đợi.
2. Khi hàng đợi chưa rỗng: lấy `u` ra (`popleft`), với mỗi `v` có `M[u][v] == 1` mà chưa thăm thì **đánh dấu ngay lúc đưa vào hàng đợi** và `append(v)`.

Điểm tinh tế: đánh dấu **lúc đưa vào** (không phải lúc lấy ra) để một đỉnh không bị xếp hàng nhiều lần.

In [12]:
def count_bfs(M: List[List[int]]) -> int:
    n = len(M)
    vis = [False] * n
    ans = 0
    for s in range(n):
        if vis[s]:
            continue
        ans += 1
        vis[s] = True
        q = deque([s])
        while q:
            u = q.popleft()
            for v in range(n):
                if M[u][v] and not vis[v]:
                    vis[v] = True
                    q.append(v)
    return ans


print(count_bfs(ex1), count_bfs(ex2), count_bfs(M6))

2 3 3


In [13]:
# Code nộp LeetCode cho cách 3 (BFS)
from collections import deque


class SolutionBFS:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        vis = [False] * n
        ans = 0
        for s in range(n):
            if vis[s]:
                continue
            ans += 1
            vis[s] = True
            q = deque([s])
            while q:
                u = q.popleft()
                for v, x in enumerate(isConnected[u]):
                    if x and not vis[v]:
                        vis[v] = True
                        q.append(v)
        return ans


print(SolutionBFS().findCircleNum(ex1), SolutionBFS().findCircleNum(ex2))

2 3


**Độ phức tạp:** mỗi đỉnh vào hàng đợi đúng một lần, mỗi lần đọc một dòng `n` ô → `O(n^2)` thời gian, `O(n)` bộ nhớ. Ưu điểm so với DFS: **không có đệ quy**, không lo tràn ngăn xếp nếu `n` lớn hơn nhiều.

### 7.3. Cách 4 — Union-Find (DSU)

Ý tưởng khác hẳn: không loang. Khởi tạo `n` nhóm (mỗi thành phố một nhóm), `ans = n`. Duyệt các ô **nửa trên** (`j > i`, vì ma trận đối xứng nên nửa dưới trùng lặp); với mỗi `M[i][j] == 1`, tìm gốc của `i` và `j`:

- Nếu **khác gốc** → hai nhóm khác nhau thật sự được nối: `union`, rồi `ans -= 1` (hai nhóm thành một).
- Nếu **cùng gốc** → đã cùng nhóm rồi, **không** giảm `ans`.

Ta cải thiện thêm bằng **union by size** (nối cây nhỏ vào cây lớn) kèm nén đường đi để cây luôn thấp.

In [14]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.size = [1] * n
        self.count = n               # số nhóm hiện có

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        self.size[ra] += self.size[rb]
        self.count -= 1
        return True


def count_dsu(M: List[List[int]]) -> int:
    n = len(M)
    d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if M[i][j]:
                d.union(i, j)
    return d.count


print(count_dsu(ex1), count_dsu(ex2), count_dsu(M6))

2 3 3


In [15]:
# Code nộp LeetCode cho cách 4 (Union-Find)
class SolutionDSU:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        p = list(range(n))

        def find(x: int) -> int:
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        ans = n
        for i in range(n):
            for j in range(i + 1, n):
                if isConnected[i][j]:
                    pa, pb = find(i), find(j)
                    if pa != pb:
                        p[pa] = pb
                        ans -= 1
        return ans


print(SolutionDSU().findCircleNum(ex1), SolutionDSU().findCircleNum(ex2))

2 3


**Độ phức tạp:** đọc `n(n-1)/2` ô là `O(n^2)`; mỗi `find` gần như hằng số (`O(alpha(n))`, hàm Ackermann ngược, coi như `<= 4` trong thực tế). Tổng `O(n^2 * alpha(n))`, bộ nhớ `O(n)`. Lưu ý: nếu chỉ nén đường đi mà không union by size, chặn trên phân tích đơn giản là `O(log n)` mỗi `find`; trong thực tế vẫn rất nhanh.

Điểm mạnh của Union-Find: **không cần đồ thị tĩnh**; ta có thể nhận cạnh theo từng đợt và hỏi "hiện có bao nhiêu nhóm" bất kỳ lúc nào (xem biến thể ở mục 15).

### 7.4. Cách 5 — Bao đóng bắc cầu (Warshall) + đếm dòng khác nhau

Quan hệ "nối trực tiếp" **chưa** bắc cầu. Ta biến nó thành quan hệ "đến được" (reachability) bằng thuật toán **Floyd–Warshall dạng boolean**: với từng thành phố trung gian `k`, với mọi cặp `(i, j)`, nếu `i` đến được `k` và `k` đến được `j` thì `i` đến được `j`.

```
for k in range(n):
    for i in range(n):
        if reach[i][k]:
            for j in range(n):
                if reach[k][j]:
                    reach[i][j] = 1
```

Sau khi xong, `reach[i][j] = 1` khi và chỉ khi `i` và `j` **cùng tỉnh**, và dòng `i` chính là **tập thành phố của tỉnh `i`**. Hai thành phố cùng tỉnh có dòng **giống hệt nhau**; khác tỉnh thì dòng khác nhau (rời nhau). Vậy đáp án = **số dòng khác nhau**.

Để gọn và nhanh, ta nén mỗi dòng thành **một số nguyên bitmask** (bit `j` bằng 1 nếu `reach[i][j] = 1`). Khi đó "`reach[i] |= reach[k]`" thay cả vòng `for j`.

In [16]:
def count_warshall_slow(M: List[List[int]]) -> int:
    """Bản dễ hiểu: O(n^3) với list 0/1."""
    n = len(M)
    reach = [row[:] for row in M]
    for k in range(n):
        for i in range(n):
            if reach[i][k]:
                for j in range(n):
                    if reach[k][j]:
                        reach[i][j] = 1
    return len({tuple(row) for row in reach})


def count_warshall_bits(M: List[List[int]]) -> int:
    """Bản bitmask: mỗi dòng là một số nguyên."""
    n = len(M)
    rows = []
    for i in range(n):
        mask = 0
        for j in range(n):
            if M[i][j]:
                mask |= 1 << j
        rows.append(mask)
    for k in range(n):
        bit_k = 1 << k
        for i in range(n):
            if rows[i] & bit_k:
                rows[i] |= rows[k]
    return len(set(rows))


print(count_warshall_slow(ex1), count_warshall_slow(ex2), count_warshall_slow(M6))
print(count_warshall_bits(ex1), count_warshall_bits(ex2), count_warshall_bits(M6))

2 3 3
2 3 3


In [17]:
# Code nộp LeetCode cho cách 5 (bao đóng bằng bitmask)
class SolutionWarshall:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        rows = [sum(1 << j for j, x in enumerate(row) if x) for row in isConnected]
        for k in range(n):
            bit_k = 1 << k
            for i in range(n):
                if rows[i] & bit_k:
                    rows[i] |= rows[k]
        return len(set(rows))


print(SolutionWarshall().findCircleNum(ex1), SolutionWarshall().findCircleNum(ex2))

2 3


**Độ phức tạp:** bản list: `O(n^3)` thời gian, `O(n^2)` bộ nhớ. Bản bitmask: `O(n^2 * n / w)` với `w` = độ rộng từ máy (ở đây số nguyên lớn của Python xử lý theo từng khối 30 bit) — với `n = 200` rất nhanh. Ý nghĩa thực sự của cách này: nó dùng một **ý tưởng độc lập** (bắc cầu thuần túy, không loang, không cây cha) nên rất hợp để làm **oracle** kiểm tra các cách còn lại.

## 8. TRACE từng bước

### 8.1. Trace DFS trên ví dụ 1

`isConnected = [[1,1,0],[1,1,0],[0,0,1]]`

| Bước | Vòng ngoài `i` | `vis` trước | Hành động | `ans` | `vis` sau |
|---|---|---|---|---|---|
| 1 | 0 | `[F,F,F]` | chưa thăm → `dfs(0)` rồi thấy bạn `1` → `dfs(1)` | 1 | `[T,T,F]` |
| 2 | 1 | `[T,T,F]` | đã thăm → bỏ qua | 1 | `[T,T,F]` |
| 3 | 2 | `[T,T,F]` | chưa thăm → `dfs(2)` (không có bạn khác) | 2 | `[T,T,T]` |

Hàm `trace_dfs` bên dưới in từng lần vào/ra `dfs` với thụt lề theo độ sâu.

In [18]:
def trace_dfs(M):
    n = len(M)
    vis = [False] * n
    ans = 0

    def dfs(i, depth):
        pad = "    " * depth
        vis[i] = True
        print(f"{pad}vào dfs({i})  vis={vis}")
        for j in range(n):
            if M[i][j] and not vis[j]:
                print(f"{pad}  {i} nối {j} và {j} chưa thăm -> đi tiếp")
                dfs(j, depth + 1)
        print(f"{pad}ra dfs({i})")

    for i in range(n):
        if vis[i]:
            print(f"i={i}: đã thăm -> bỏ qua")
        else:
            ans += 1
            print(f"i={i}: CHƯA thăm -> tỉnh mới, ans = {ans}")
            dfs(i, 1)
    print("kết quả:", ans)
    return ans


trace_dfs(ex1)

i=0: CHƯA thăm -> tỉnh mới, ans = 1
    vào dfs(0)  vis=[True, False, False]
      0 nối 1 và 1 chưa thăm -> đi tiếp
        vào dfs(1)  vis=[True, True, False]
        ra dfs(1)
    ra dfs(0)
i=1: đã thăm -> bỏ qua
i=2: CHƯA thăm -> tỉnh mới, ans = 2
    vào dfs(2)  vis=[True, True, True]
    ra dfs(2)
kết quả: 2


2

### 8.2. Trace DFS trên đồ thị 6 thành phố (`0-2`, `2-4`, `1-3`)

Ở đồ thị này `dfs(0)` đi `0 -> 2 -> 4` (độ sâu 3) rồi quay lui; sau đó `1 -> 3`; và `5` đứng một mình.

In [19]:
trace_dfs(M6)

i=0: CHƯA thăm -> tỉnh mới, ans = 1
    vào dfs(0)  vis=[True, False, False, False, False, False]
      0 nối 2 và 2 chưa thăm -> đi tiếp
        vào dfs(2)  vis=[True, False, True, False, False, False]
          2 nối 4 và 4 chưa thăm -> đi tiếp
            vào dfs(4)  vis=[True, False, True, False, True, False]
            ra dfs(4)
        ra dfs(2)
    ra dfs(0)
i=1: CHƯA thăm -> tỉnh mới, ans = 2
    vào dfs(1)  vis=[True, True, True, False, True, False]
      1 nối 3 và 3 chưa thăm -> đi tiếp
        vào dfs(3)  vis=[True, True, True, True, True, False]
        ra dfs(3)
    ra dfs(1)
i=2: đã thăm -> bỏ qua
i=3: đã thăm -> bỏ qua
i=4: đã thăm -> bỏ qua
i=5: CHƯA thăm -> tỉnh mới, ans = 3
    vào dfs(5)  vis=[True, True, True, True, True, True]
    ra dfs(5)
kết quả: 3


3

### 8.3. Trace Union-Find trên đồ thị 6 thành phố

Các cạnh nửa trên theo thứ tự quét `i` rồi `j`: `(0,2)`, `(1,3)`, `(2,4)`. Bảng dưới (union by size; khi bằng cỡ, gốc của `a` giữ vai trò gốc):

| Cạnh | gốc a, gốc b | Có gộp? | `p` sau | `ans` |
|---|---|---|---|---|
| (khởi tạo) | | | `[0,1,2,3,4,5]` | 6 |
| (0,2) | 0, 2 | có | `[0,1,0,3,4,5]` | 5 |
| (1,3) | 1, 3 | có | `[0,1,0,1,4,5]` | 4 |
| (2,4) | 0, 4 | có | `[0,1,0,1,0,5]` | 3 |

In [20]:
def trace_dsu(M):
    n = len(M)
    d = DSU(n)
    print("khởi tạo  p =", d.p, " ans =", d.count)
    for i in range(n):
        for j in range(i + 1, n):
            if M[i][j]:
                ra, rb = d.find(i), d.find(j)
                merged = d.union(i, j)
                print(f"cạnh ({i},{j}) gốc {ra},{rb} -> {'gộp' if merged else 'đã cùng nhóm'}"
                      f"  p = {d.p}  ans = {d.count}")
    return d.count


print("kết quả:", trace_dsu(M6))

khởi tạo  p = [0, 1, 2, 3, 4, 5]  ans = 6
cạnh (0,2) gốc 0,2 -> gộp  p = [0, 1, 0, 3, 4, 5]  ans = 5
cạnh (1,3) gốc 1,3 -> gộp  p = [0, 1, 0, 1, 4, 5]  ans = 4
cạnh (2,4) gốc 0,4 -> gộp  p = [0, 1, 0, 1, 0, 5]  ans = 3
kết quả: 3


In [21]:
# Một đồ thị có chu trình để thấy nhánh "đã cùng nhóm" (không giảm ans)
M_cycle = [[1, 1, 1], [1, 1, 1], [1, 1, 1]]     # tam giác 0-1-2
print("kết quả:", trace_dsu(M_cycle))

khởi tạo  p = [0, 1, 2]  ans = 3
cạnh (0,1) gốc 0,1 -> gộp  p = [0, 0, 2]  ans = 2
cạnh (0,2) gốc 0,2 -> gộp  p = [0, 0, 0]  ans = 1
cạnh (1,2) gốc 0,0 -> đã cùng nhóm  p = [0, 0, 0]  ans = 1
kết quả: 1


## 9. Code nộp LeetCode

Ta chọn **DFS** làm lời giải nộp: ngắn, dễ chứng minh, `O(n^2)` thời gian, `O(n)` bộ nhớ, và với `n <= 200` độ sâu đệ quy an toàn. (Union-Find ở 7.3 cũng đúng và tương đương.)

In [22]:
class Solution:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        vis = [False] * n

        def dfs(i: int) -> None:
            vis[i] = True
            for j, x in enumerate(isConnected[i]):
                if x and not vis[j]:
                    dfs(j)

        ans = 0
        for i in range(n):
            if not vis[i]:
                dfs(i)
                ans += 1
        return ans


sol = Solution()
print(sol.findCircleNum(ex1))   # 2
print(sol.findCircleNum(ex2))   # 3

2
3


## 10. Vì sao đúng

Gọi quan hệ `i ~ j` là "`i` đến được `j` qua các cạnh". Vì ma trận đối xứng và có đường chéo bằng 1, `~` có tính **phản xạ** (`i ~ i`), **đối xứng** (`i ~ j` thì `j ~ i`) và **bắc cầu** (đi `i -> j -> k`). Vậy `~` là quan hệ tương đương nên chia tập thành phố thành các lớp rời nhau — đó chính là các tỉnh.

**Tính đúng của DFS/BFS (đếm).** Xét lúc vòng ngoài gặp `i` chưa thăm và gọi loang từ `i`.

- *Loang vét đủ tỉnh của `i`*: nếu `u` đã thăm và `M[u][v] = 1` thì `v` sẽ bị thăm (hoặc đã thăm). Quy nạp theo độ dài đường đi từ `i`: mọi thành phố đến được từ `i` đều bị thăm.
- *Loang không tràn ra ngoài tỉnh*: ta chỉ đi theo cạnh, nên mọi đỉnh bị thăm đều đến được từ `i`.
- Vậy sau mỗi lần loang, **đúng một tỉnh** chuyển sang "đã thăm hoàn toàn". Thành phố của tỉnh đó không bao giờ kích hoạt lần đếm nữa. Một tỉnh có thành phố nhỏ nhất `m`: khi vòng ngoài đến `m` thì chưa có thành phố nào của tỉnh bị thăm (vì chỉ loang từ tỉnh khác thì không chạm tới) nên **đúng một** lần đếm cho tỉnh đó.

**Tính đúng của Union-Find.** Mỗi lần `union` thành công, số nhóm giảm đúng 1; lúc cuối, mỗi nhóm của DSU chính xác là một tập các đỉnh nối nhau qua các cạnh đã xét, tức một tỉnh. `ans = n - (số lần gộp thành công)`.

**Tính đúng của Warshall.** Sau khi xử lý trung gian `k = 0..K`, `reach[i][j] = 1` khi tồn tại đường đi từ `i` đến `j` chỉ qua các trung gian trong `{0..K}`. Khi `K = n - 1`, nó là bao đóng bắc cầu đầy đủ.

In [23]:
# Kiểm tra bất biến "ans = n - số lần union thành công" trên một đồ thị ngẫu nhiên nhỏ
rng = random.Random(1)
n = 12
M = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
for _ in range(8):
    a, b = rng.randrange(n), rng.randrange(n)
    M[a][b] = M[b][a] = 1

d = DSU(n)
ok = 0
for i in range(n):
    for j in range(i + 1, n):
        if M[i][j] and d.union(i, j):
            ok += 1
print("n =", n, "| số union thành công =", ok, "| n - ok =", n - ok, "| d.count =", d.count)
assert d.count == n - ok == count_dfs(M)

n = 12 | số union thành công = 6 | n - ok = 6 | d.count = 6


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Gộp nhóm bằng set | `O(n^3)` tệ nhất | `O(n)` |
| 2. DFS đệ quy | `O(n^2)` | `O(n)` (ngăn xếp đệ quy) |
| 3. BFS | `O(n^2)` | `O(n)` (hàng đợi) |
| 4. Union-Find | `O(n^2 * alpha(n))` | `O(n)` |
| 5. Warshall bitmask | `O(n^3 / w)` | `O(n)` bit-rows (ở dạng list 0/1 là `O(n^2)`) |

Vì sao `O(n^2)` là **tối ưu** cho đầu vào ma trận? Muốn chắc chắn không bỏ sót cạnh nào, ta phải đọc ma trận; kích thước đầu vào đã là `n^2` ô. (Nếu đề cho danh sách cạnh thì DSU `O(E * alpha)` sẽ có lợi.)

In [24]:
# Đo số lần đọc ô ma trận của DFS và DSU: cả hai ~ n^2 (DSU ~ n^2/2)
def reads_dfs(M):
    n = len(M); vis = [False] * n; reads = 0
    def dfs(i):
        nonlocal reads
        vis[i] = True
        for j in range(n):
            reads += 1
            if M[i][j] and not vis[j]:
                dfs(j)
    for i in range(n):
        if not vis[i]:
            dfs(i)
    return reads


def reads_dsu(M):
    n = len(M)
    return sum(1 for i in range(n) for j in range(i + 1, n))


for n in (10, 50, 100, 200):
    M = [[1 if i == j or (i + j) % 7 == 0 else 0 for j in range(n)] for i in range(n)]
    # làm đối xứng (i + j) % 7 đã đối xứng sẵn
    print(f"n={n:3d}  DFS đọc {reads_dfs(M):6d} ô | DSU đọc {reads_dsu(M):6d} ô | n^2 = {n * n}")

n= 10  DFS đọc    100 ô | DSU đọc     45 ô | n^2 = 100
n= 50  DFS đọc   2500 ô | DSU đọc   1225 ô | n^2 = 2500
n=100  DFS đọc  10000 ô | DSU đọc   4950 ô | n^2 = 10000
n=200  DFS đọc  40000 ô | DSU đọc  19900 ô | n^2 = 40000


## 12. Bộ test

Chiến lược: (a) ví dụ đề; (b) edge case; (c) phản ví dụ cho các cách sai; (d) đối chiếu 5 cách + oracle độc lập trên nhiều input; (e) random có seed **có đáp án đã biết trước** (sinh đồ thị có đúng `k` tỉnh).

In [25]:
solvers = {
    "gộp set": count_groups_merge,
    "DFS": count_dfs,
    "BFS": count_bfs,
    "DSU": count_dsu,
    "Warshall list": count_warshall_slow,
    "Warshall bit": count_warshall_bits,
    "nộp DFS": lambda M: Solution().findCircleNum(M),
    "nộp BFS": lambda M: SolutionBFS().findCircleNum(M),
    "nộp DSU": lambda M: SolutionDSU().findCircleNum(M),
    "nộp Warshall": lambda M: SolutionWarshall().findCircleNum(M),
}


def check_all(M, expected, label=""):
    for name, f in solvers.items():
        got = f([row[:] for row in M])
        assert got == expected, (label, name, got, expected)


# (a) ví dụ đề
check_all(ex1, 2, "ví dụ 1")
check_all(ex2, 3, "ví dụ 2")
print("ví dụ đề: OK")

ví dụ đề: OK


In [26]:
# (b) edge case
def identity(n):
    return [[1 if i == j else 0 for j in range(n)] for i in range(n)]


def complete(n):
    return [[1] * n for _ in range(n)]


def chain(n):
    M = identity(n)
    for i in range(n - 1):
        M[i][i + 1] = M[i + 1][i] = 1
    return M


def star(n):
    M = identity(n)
    for i in range(1, n):
        M[0][i] = M[i][0] = 1
    return M


check_all([[1]], 1, "n=1")
check_all(complete(2), 1, "n=2 nối nhau")
check_all(identity(2), 2, "n=2 rời nhau")
check_all(complete(10), 1, "đồ thị đầy đủ")
check_all(identity(10), 10, "toàn cô lập")
check_all(chain(30), 1, "một đường thẳng 30 đỉnh")
check_all(star(30), 1, "hình sao 30 đỉnh")

# hai khối chéo: {0..4} và {5..9}
two = identity(10)
for blk in (range(0, 5), range(5, 10)):
    for a in blk:
        for b in blk:
            two[a][b] = 1
check_all(two, 2, "hai khối")
print("edge case: OK")

edge case: OK


In [27]:
# (c) phản ví dụ: hai cách ngây thơ sai ở đâu
def wrong_count_distinct_rows(M):
    """Đếm số dòng khác nhau (SAI vì không xét bắc cầu)."""
    return len({tuple(r) for r in M})


def wrong_n_minus_edges(M):
    """n - số cạnh (SAI khi có chu trình)."""
    n = len(M)
    edges = sum(M[i][j] for i in range(n) for j in range(i + 1, n))
    return n - edges


print("chain(3):", chain(3))
print("đúng =", count_dfs(chain(3)), "| đếm dòng khác nhau =", wrong_count_distinct_rows(chain(3)))
print("tam giác: đúng =", count_dfs(complete(3)), "| n - cạnh =", wrong_n_minus_edges(complete(3)))
assert wrong_count_distinct_rows(chain(3)) != count_dfs(chain(3))
assert wrong_n_minus_edges(complete(3)) != count_dfs(complete(3))

chain(3): [[1, 1, 0], [1, 1, 1], [0, 1, 1]]
đúng = 1 | đếm dòng khác nhau = 3
tam giác: đúng = 1 | n - cạnh = 0


In [28]:
# (d) đối chiếu nhiều input nhỏ: vét cạn mọi ma trận đối xứng có đường chéo 1 với n <= 5
def all_matrices(n):
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    for mask in range(1 << len(pairs)):
        M = identity(n)
        for b, (i, j) in enumerate(pairs):
            if mask >> b & 1:
                M[i][j] = M[j][i] = 1
        yield M


def oracle_label_propagation(M):
    """Oracle độc lập: gán nhãn nhỏ nhất, lặp tới khi hết thay đổi."""
    n = len(M)
    label = list(range(n))
    changed = True
    while changed:
        changed = False
        for i in range(n):
            for j in range(n):
                if M[i][j] and label[j] < label[i]:
                    label[i] = label[j]
                    changed = True
    return len(set(label))


total = 0
for n in range(1, 6):
    for M in all_matrices(n):
        expected = oracle_label_propagation(M)
        check_all(M, expected, f"n={n}")
        total += 1
print("đã đối chiếu", total, "ma trận (n = 1..5, vét cạn): OK")

đã đối chiếu 1099 ma trận (n = 1..5, vét cạn): OK


In [29]:
# (e) random có seed, sinh đồ thị có ĐÚNG k tỉnh (đáp án biết trước)
def planted(n, k, rng, extra=0.3):
    order = list(range(n))
    rng.shuffle(order)
    groups = [[] for _ in range(k)]
    for idx, city in enumerate(order):
        g = idx if idx < k else rng.randrange(k)       # k thành phố đầu mỗi nhóm một người
        groups[g].append(city)
    M = identity(n)
    for members in groups:
        for t in range(1, len(members)):                # cây ngẫu nhiên nối cả nhóm
            a = members[t]
            b = members[rng.randrange(t)]
            M[a][b] = M[b][a] = 1
        for a in members:                               # thêm cạnh thừa trong nhóm
            for b in members:
                if a < b and rng.random() < extra:
                    M[a][b] = M[b][a] = 1
    return M


rng = random.Random(2024)
for case in range(300):
    n = rng.randint(1, 40)
    k = rng.randint(1, n)
    M = planted(n, k, rng)
    assert all(M[i][i] == 1 for i in range(n))
    assert all(M[i][j] == M[j][i] for i in range(n) for j in range(n))
    check_all(M, k, f"planted case {case}")
print("300 ca ngẫu nhiên có đáp án biết trước: OK")

300 ca ngẫu nhiên có đáp án biết trước: OK


In [30]:
# (f) cỡ lớn nhất n = 200, đo thời gian lời giải nộp
rng = random.Random(7)
for k in (1, 2, 50, 200):
    M = planted(200, k, rng, extra=0.05)
    t0 = time.perf_counter()
    got = Solution().findCircleNum(M)
    dt = time.perf_counter() - t0
    assert got == k
    assert SolutionDSU().findCircleNum(M) == SolutionBFS().findCircleNum(M) == SolutionWarshall().findCircleNum(M) == k
    print(f"n=200, k={k:3d}: đáp án {got:3d}, DFS mất {dt * 1000:.1f} ms")
M = chain(200)   # độ sâu đệ quy tệ nhất
assert Solution().findCircleNum(M) == 1
print("chuỗi 200 đỉnh (đệ quy sâu 200): OK")

n=200, k=  1: đáp án   1, DFS mất 3.0 ms
n=200, k=  2: đáp án   2, DFS mất 1.3 ms
n=200, k= 50: đáp án  50, DFS mất 1.2 ms
n=200, k=200: đáp án 200, DFS mất 1.2 ms
chuỗi 200 đỉnh (đệ quy sâu 200): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số dòng khác nhau của ma trận (quên bắc cầu)

Thoạt nghĩ: "cùng tỉnh thì dòng giống nhau" nên đếm số dòng khác nhau. Sai: dòng của `0` là "bạn trực tiếp của 0", không chứa bạn gián tiếp.

In [31]:
def wrong_rows(isConnected):
    return len({tuple(r) for r in isConnected})


M = chain(3)           # 0-1, 1-2: một tỉnh
print_matrix(M)
print("đúng (DFS):", count_dfs(M), "| cách sai (đếm dòng khác nhau):", wrong_rows(M))

1 1 0
1 1 1
0 1 1
đúng (DFS): 1 | cách sai (đếm dòng khác nhau): 3


**Vì sao sai:** dòng 0 là `1 1 0`, dòng 1 là `1 1 1`, dòng 2 là `0 1 1` — cả ba khác nhau, dù thực ra cùng một tỉnh. Ma trận mới chỉ có quan hệ **trực tiếp**; ta phải loang/gộp để có quan hệ **gián tiếp**. (Chỉ đếm dòng khác nhau **sau khi** đã lấy bao đóng bắc cầu — chính là cách 5.)

### Lỗi 2 — Quên `visited` (hoặc kiểm tra sai chỗ) dẫn đến đệ quy vô hạn

In [32]:
import sys


def wrong_dfs_no_vis(M):
    n = len(M)

    def dfs(i):
        for j in range(n):
            if M[i][j] and j != i:      # thiếu `not vis[j]`
                dfs(j)

    ans = 0
    dfs(0)
    return ans


try:
    wrong_dfs_no_vis([[1, 1], [1, 1]])
except RecursionError:
    print("RecursionError: 0 -> 1 -> 0 -> 1 -> ... không bao giờ dừng")

RecursionError: 0 -> 1 -> 0 -> 1 -> ... không bao giờ dừng


**Cách sửa:** đánh dấu `vis[i] = True` ngay khi vào `dfs(i)` và chỉ đi vào đỉnh chưa thăm (`not vis[j]`) — đúng như `Solution` ở mục 9. (Đoạn trên bắt `RecursionError` để notebook không dừng; trên LeetCode đây là lỗi Runtime Error.)

### Lỗi 3 — Giảm bộ đếm mỗi lần thấy cạnh, kể cả khi đã cùng nhóm (Union-Find)

Người mới thường viết `ans -= 1` cho mọi `M[i][j] == 1`. Khi có chu trình (tam giác), bộ đếm bị trừ quá tay.

In [33]:
def wrong_dsu(M):
    n = len(M)
    p = list(range(n))

    def find(x):
        while p[x] != x:
            x = p[x]
        return x

    ans = n
    for i in range(n):
        for j in range(i + 1, n):
            if M[i][j]:
                p[find(i)] = find(j)
                ans -= 1               # SAI: trừ cả khi đã cùng nhóm
    return ans


tri = complete(3)
print("tam giác: đúng =", count_dfs(tri), "| bản sai =", wrong_dsu(tri))

tam giác: đúng = 1 | bản sai = 0


**Vì sao sai:** tam giác `0-1-2` có 3 cạnh nhưng chỉ cần 2 cạnh để nối 3 đỉnh; cạnh thứ 3 nối hai đỉnh **đã cùng gốc**, không làm giảm số nhóm. Sửa: chỉ `ans -= 1` khi `find(i) != find(j)`.

### Lỗi 4 — Bắt đầu duyệt mà không bọc vòng ngoài (chỉ loang từ thành phố 0)

Chỉ gọi `dfs(0)` rồi trả về 1 sẽ sai khi có nhiều tỉnh; ta phải thử **từng** thành phố chưa thăm.

In [34]:
def wrong_only_from_zero(M):
    n = len(M)
    vis = [False] * n

    def dfs(i):
        vis[i] = True
        for j in range(n):
            if M[i][j] and not vis[j]:
                dfs(j)

    dfs(0)
    return 1               # SAI: giả sử chỉ có một tỉnh


print("ví dụ 1: đúng =", count_dfs(ex1), "| bản sai =", wrong_only_from_zero(ex1))
print("ví dụ 2: đúng =", count_dfs(ex2), "| bản sai =", wrong_only_from_zero(ex2))

ví dụ 1: đúng = 2 | bản sai = 1
ví dụ 2: đúng = 3 | bản sai = 1


**Cách sửa:** vòng ngoài `for i in range(n): if not vis[i]: dfs(i); ans += 1`.

### Lỗi 5 — Đánh dấu thăm lúc `popleft` thay vì lúc `append` (BFS)

Không sai kết quả nhưng một đỉnh có thể bị xếp hàng nhiều lần, làm chậm và có thể phình hàng đợi. Cell dưới đếm số lần đưa vào hàng đợi.

In [35]:
def bfs_pushes(M, mark_on_push):
    n = len(M)
    vis = [False] * n
    pushes = 0
    ans = 0
    for s in range(n):
        if vis[s]:
            continue
        ans += 1
        q = deque([s])
        pushes += 1
        if mark_on_push:
            vis[s] = True
        while q:
            u = q.popleft()
            if not mark_on_push:
                if vis[u]:
                    continue
                vis[u] = True
            for v in range(n):
                if M[u][v] and not vis[v]:
                    if mark_on_push:
                        vis[v] = True
                    q.append(v)
                    pushes += 1
    return ans, pushes


K = complete(30)
print("đánh dấu lúc đưa vào hàng đợi:", bfs_pushes(K, True))
print("đánh dấu lúc lấy ra          :", bfs_pushes(K, False))

đánh dấu lúc đưa vào hàng đợi: (1, 30)
đánh dấu lúc lấy ra          : (1, 436)


**Cách sửa:** đánh dấu `vis[v] = True` **ngay khi `append(v)`** (như cách 3), để mỗi đỉnh vào hàng đợi đúng một lần.

## 14. Mẹo & nhận dạng pattern

- Thấy "nhóm", "cụm", "nối trực tiếp hoặc gián tiếp", "bao nhiêu thành phần/khối/đảo" → nghĩ ngay đến **thành phần liên thông**: DFS/BFS hoặc Union-Find.
- Đề cho **ma trận kề** `n x n`: duyệt dòng `i` là duyệt "bạn của `i`". Đề cho **danh sách cạnh** `edges`: dùng Union-Find (khỏi dựng đồ thị), hoặc dựng danh sách kề rồi DFS/BFS.
- Chọn DFS hay BFS: kết quả giống nhau. DFS đệ quy ngắn nhất; BFS/ngăn xếp an toàn khi đồ thị rất sâu (Python giới hạn đệ quy ~1000).
- Khi bài **tăng dần cạnh** hoặc hỏi online "bây giờ có bao nhiêu nhóm?" → Union-Find (DFS phải chạy lại từ đầu).
- Công thức hay dùng: `số thành phần liên thông = n - số lần union thành công`; với rừng (không chu trình) `= n - số cạnh`.
- Với ma trận đối xứng, chỉ cần xét nửa trên (`j > i`) cho Union-Find; DFS/BFS đọc cả dòng.
- Gắn nhãn kiểm tra nhanh: `isConnected[i][i] == 1` luôn đúng — đừng đếm nó là "cạnh".

## 15. Biến thể

### 15.1. Liệt kê chính các tỉnh (không chỉ đếm)

Mỗi lần loang, gom các đỉnh vào một list.

In [36]:
def list_provinces(M):
    n = len(M)
    vis = [False] * n
    res = []
    for s in range(n):
        if vis[s]:
            continue
        comp = []
        vis[s] = True
        stack = [s]
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in range(n):
                if M[u][v] and not vis[v]:
                    vis[v] = True
                    stack.append(v)
        res.append(sorted(comp))
    return res


print(list_provinces(M6))          # [[0, 2, 4], [1, 3], [5]]
print(list_provinces(ex1))         # [[0, 1], [2]]

[[0, 2, 4], [1, 3], [5]]
[[0, 1], [2]]


### 15.2. Kích thước tỉnh lớn nhất

Chỉ cần lấy `max(len(c) for c in list_provinces(M))`; hoặc dùng `DSU.size` ở cách 4.

In [37]:
def largest_province(M):
    return max(len(c) for c in list_provinces(M))


def largest_province_dsu(M):
    n = len(M)
    d = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if M[i][j]:
                d.union(i, j)
    return max(d.size[d.find(i)] for i in range(n))


print(largest_province(M6), largest_province_dsu(M6))     # 3 3
assert largest_province(two) == largest_province_dsu(two) == 5

3 3


### 15.3. Đầu vào là danh sách cạnh (LeetCode 323 — Number of Connected Components in an Undirected Graph)

Cho `n` và `edges = [[a, b], ...]`; đếm thành phần liên thông. Dùng Union-Find: `ans = n`, mỗi cạnh nối hai gốc khác nhau thì `ans -= 1`.

In [38]:
def count_components_edges(n, edges):
    d = DSU(n)
    for a, b in edges:
        d.union(a, b)
    return d.count


print(count_components_edges(5, [[0, 1], [1, 2], [3, 4]]))        # 2
print(count_components_edges(5, [[0, 1], [1, 2], [2, 3], [3, 4]]))  # 1

2
1


### 15.4. Cạnh đến dần theo thời gian (online)

Với Union-Find, sau mỗi cạnh mới ta đọc ngay `d.count`. Mỗi cạnh `O(alpha(n))` — DFS thì phải làm lại từ đầu.

In [39]:
def components_over_time(n, edges):
    d = DSU(n)
    out = []
    for a, b in edges:
        d.union(a, b)
        out.append(d.count)
    return out


print(components_over_time(6, [(0, 2), (1, 3), (2, 4), (0, 4), (4, 5)]))   # [5, 4, 3, 3, 2]

[5, 4, 3, 3, 2]


### 15.5. Các bài họ hàng

- LeetCode 200 *Number of Islands*: đếm thành phần liên thông trên lưới (đỉnh = ô).
- LeetCode 323 (ở trên), 684 *Redundant Connection* (cạnh nào làm xuất hiện chu trình — Union-Find trả `False`), 1319 *Number of Operations to Make Network Connected* (đáp án `components - 1` nếu đủ cáp).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gộp nhóm set | Duyệt cạnh, gộp các set giao nhau | `O(n^3)` | `O(n)` | Dễ nghĩ, để hiểu bài; không nên nộp với `n` lớn |
| 2. DFS đệ quy | Loang từ thành phố chưa thăm | `O(n^2)` | `O(n)` | Mặc định, ngắn gọn; `n` nhỏ/vừa |
| 3. BFS | Loang bằng hàng đợi | `O(n^2)` | `O(n)` | Tránh đệ quy sâu; muốn khoảng cách ngắn nhất kèm theo |
| 4. Union-Find | Gộp nhóm, đếm số lần gộp thật | `O(n^2 * alpha(n))` | `O(n)` | Đầu vào là danh sách cạnh, cạnh đến dần (online) |
| 5. Warshall bitmask | Bao đóng bắc cầu + đếm dòng khác nhau | `O(n^3 / w)` | `O(n)` | Oracle kiểm tra chéo, hoặc cần cả ma trận "đến được" |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(count_dfs([[1,1,1],[1,1,0],[1,0,1]]))
print(count_dfs([[1,0],[0,1]]))
print(list_provinces([[1,0,1],[0,1,0],[1,0,1]]))
```

In [40]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Cho ma trận `5 x 5` với các cạnh `0-1`, `1-2`, `3-4`, `0-2`. Dùng Union-Find, quét cạnh theo thứ tự `(0,1)`, `(0,2)`, `(1,2)`, `(3,4)` và ghi `ans` sau mỗi cạnh. Hỏi: đáp án cuối là gì, và cạnh nào **không** làm giảm `ans`?

In [41]:
# Bài 2: điền kết quả tính tay
hand_ans_after_each_edge = None    # ví dụ: [4, 3, ...] (4 số)
hand_final_answer = None
hand_edge_not_reducing = None      # ví dụ: (0, 1)

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm đếm số tỉnh bằng DFS dùng **ngăn xếp** (không đệ quy).

In [42]:
def exercise_3_stack(M: List[List[int]]) -> int:
    n = len(M)
    vis = [False] * n
    ans = 0
    for s in range(n):
        # TODO 1: nếu s đã thăm thì bỏ qua
        # TODO 2: tăng ans, đánh dấu s, đẩy s vào stack
        # TODO 3: while stack: lấy u ra; với mỗi v là bạn của u mà chưa thăm -> đánh dấu và đẩy vào stack
        pass
    return ans


# assert exercise_3_stack(ex1) == 2
# assert exercise_3_stack(ex2) == 3
# assert exercise_3_stack(chain(20)) == 1

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_dsu(M)` bằng Union-Find (đủ `find` có nén đường đi và bộ đếm nhóm).

In [43]:
def exercise_4_dsu(M: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_dsu(ex1) == 2
# assert exercise_4_dsu(complete(5)) == 1
# assert exercise_4_dsu(identity(5)) == 5

### Bài 5 — Mini challenge: số cáp tối thiểu để nối mọi tỉnh (khó)

Hiện mạng gồm các tỉnh như đề. Mỗi **sợi cáp mới** nối được hai thành phố bất kỳ. Tính số sợi cáp mới **tối thiểu** để cả `n` thành phố thông nhau (luôn làm được vì ta cứ nối các tỉnh lại với nhau). Viết `exercise_5_min_new_cables(M)`.

Gợi ý hướng nghĩ: nối `p` tỉnh thành một cần ít nhất `p - 1` sợi.

```python
exercise_5_min_new_cables(ex1) -> 1
exercise_5_min_new_cables(ex2) -> 2
exercise_5_min_new_cables([[1]]) -> 0
```

In [44]:
def exercise_5_min_new_cables(M: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_min_new_cables(ex1) == 1
# assert exercise_5_min_new_cables(ex2) == 2
# assert exercise_5_min_new_cables([[1]]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: Vẽ đồ thị từ ma trận, tìm các nhóm bằng mắt.
- Mức 2: Ở dòng 1, ma trận `[[1,1,1],[1,1,0],[1,0,1]]`: thành phố 0 nối cả 1 và 2.
- Mức 3: Dòng 1 → `1`; dòng 2 → `2`; dòng 3 → `[[0, 2], [1]]`.

**Bài 2**
- Mức 1: Bắt đầu `ans = 5`; mỗi lần gộp thật thì trừ 1.
- Mức 2: Sau `(0,1)` còn 4; `(0,2)` gộp `2` vào nhóm của `0`.
- Mức 3: Cạnh `(1,2)` nối hai đỉnh đã cùng gốc nên không giảm; `ans` lần lượt `[4, 3, 3, 2]`, đáp án `2`.

**Bài 3**
- Mức 1: Giống BFS nhưng thay `popleft` bằng `pop()`.
- Mức 2: Đánh dấu thăm lúc đẩy vào stack.
- Mức 3: Xem đáp án mục 19.

**Bài 4**
- Mức 1: `p = list(range(n))`, `find` đi lên đến khi `p[x] == x`.
- Mức 2: Chỉ xét `j > i`; nếu hai gốc khác nhau thì nối và giảm đếm.
- Mức 3: Xem đáp án mục 19.

**Bài 5**
- Mức 1: Cần ít nhất bao nhiêu sợi để nối `p` khối rời nhau?
- Mức 2: Mỗi sợi cáp mới giảm số khối nhiều nhất 1.
- Mức 3: Trả `số tỉnh - 1`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [45]:
print(count_dfs([[1, 1, 1], [1, 1, 0], [1, 0, 1]]))            # 1
print(count_dfs([[1, 0], [0, 1]]))                              # 2
print(list_provinces([[1, 0, 1], [0, 1, 0], [1, 0, 1]]))        # [[0, 2], [1]]

assert count_dfs([[1, 1, 1], [1, 1, 0], [1, 0, 1]]) == 1
assert count_dfs([[1, 0], [0, 1]]) == 2
assert list_provinces([[1, 0, 1], [0, 1, 0], [1, 0, 1]]) == [[0, 2], [1]]

1
2
[[0, 2], [1]]


### Đáp án Bài 2

In [46]:
M5 = identity(5)
for a, b in [(0, 1), (1, 2), (3, 4), (0, 2)]:
    M5[a][b] = M5[b][a] = 1

d = DSU(5)
history = []
for a, b in [(0, 1), (0, 2), (1, 2), (3, 4)]:
    d.union(a, b)
    history.append(d.count)
print(history, d.count)
assert history == [4, 3, 3, 2] and d.count == 2 == count_dfs(M5)

[4, 3, 3, 2] 2


Cạnh `(1,2)` **không** làm giảm `ans` vì khi đó `1` và `2` đã cùng nhóm (qua `0`).

### Đáp án Bài 3

In [47]:
def exercise_3_stack_solution(M: List[List[int]]) -> int:
    n = len(M)
    vis = [False] * n
    ans = 0
    for s in range(n):
        if vis[s]:
            continue
        ans += 1
        vis[s] = True
        stack = [s]
        while stack:
            u = stack.pop()
            for v in range(n):
                if M[u][v] and not vis[v]:
                    vis[v] = True
                    stack.append(v)
    return ans


assert exercise_3_stack_solution(ex1) == 2
assert exercise_3_stack_solution(ex2) == 3
assert exercise_3_stack_solution(chain(20)) == 1
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [48]:
def exercise_4_dsu_solution(M: List[List[int]]) -> int:
    n = len(M)
    p = list(range(n))

    def find(x):
        while p[x] != x:
            p[x] = p[p[x]]
            x = p[x]
        return x

    cnt = n
    for i in range(n):
        for j in range(i + 1, n):
            if M[i][j]:
                ra, rb = find(i), find(j)
                if ra != rb:
                    p[ra] = rb
                    cnt -= 1
    return cnt


assert exercise_4_dsu_solution(ex1) == 2
assert exercise_4_dsu_solution(complete(5)) == 1
assert exercise_4_dsu_solution(identity(5)) == 5
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Có `p` tỉnh thì cần đúng `p - 1` sợi cáp mới (mỗi sợi nối hai tỉnh khác nhau làm số tỉnh giảm 1; không thể giảm nhanh hơn vì mỗi sợi chỉ chạm tối đa hai tỉnh).

In [49]:
def exercise_5_solution(M: List[List[int]]) -> int:
    return count_dfs(M) - 1


assert exercise_5_solution(ex1) == 1
assert exercise_5_solution(ex2) == 2
assert exercise_5_solution([[1]]) == 0
assert exercise_5_solution(complete(6)) == 0
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS đếm thành phần liên thông)

```python
class Solution:
    def findCircleNum(self, isConnected: List[List[int]]) -> int:
        n = len(isConnected)
        vis = [False] * n

        def dfs(i):
            vis[i] = True
            for j, x in enumerate(isConnected[i]):
                if x and not vis[j]:
                    dfs(j)

        ans = 0
        for i in range(n):
            if not vis[i]:
                dfs(i)
                ans += 1
        return ans
```

### Template (Union-Find)

```python
p = list(range(n))
def find(x):
    while p[x] != x:
        p[x] = p[p[x]]
        x = p[x]
    return x
ans = n
for i in range(n):
    for j in range(i + 1, n):
        if M[i][j]:
            a, b = find(i), find(j)
            if a != b:
                p[a] = b
                ans -= 1
```

### Checklist trước khi submit

- [ ] Có vòng ngoài thử **mọi** thành phố, và `ans += 1` đúng chỗ (khi gặp đỉnh **chưa thăm**).
- [ ] Đánh dấu `vis` ngay khi vào đỉnh (DFS) hoặc ngay khi đưa vào hàng đợi (BFS).
- [ ] Union-Find: chỉ giảm bộ đếm khi hai gốc **khác nhau**.
- [ ] Không đếm số dòng khác nhau của ma trận khi chưa lấy bao đóng bắc cầu.
- [ ] Đã thử `n = 1`, toàn cô lập, đồ thị đầy đủ, đường thẳng dài 200 (độ sâu đệ quy).
- [ ] Signature đúng: `class Solution`, `findCircleNum(self, isConnected: List[List[int]]) -> int`.